# P1b · Estimated Migratory Balance in Spain (1998–2024)
## *Net Padrón Change minus Vegetative Balance (MNP) by Municipality Size and Goerlich (2016) Typology*
## *RURIMESCAPE — Paper 1, Step 1b*

---

**Paper:** Rural Migration and Land Use in Spain — Paper 1  
**Step:** 1b — Estimated migratory balance (Part B of three)  
**Author:** Juan Zotes  
**Last updated:** 2026-04

---

### Context and purpose

Notebook p1a showed that net padrón change for municipalities <5,000 inhabitants
becomes sustainably positive from ~2020, approximately two years later than the
2018 inflection identified by MITERD (2022) using residential flow data (EVR).
This lag is attributable to the structural negative vegetative balance of small
rural municipalities.

This notebook removes the vegetative component to isolate the estimated migratory
balance, which is directly comparable to the EVR-based measure used by MITERD:

$$\hat{M}_{i,t} = \Delta P_{i,t} - V_{i,t}$$

where $\Delta P_{i,t}$ is the annual net padrón change and $V_{i,t}$ is the
vegetative balance (births minus deaths) from the *Movimiento Natural de la
Población* (MNP, INE).

This is **Part B** of a three-notebook series:

| Notebook | Outcome variable | Source |
|----------|-----------------|--------|
| p1a | Net padrón change (ΔP) | Padrón Municipal 1996–2025 |
| **p1b** (this notebook) | Estimated migratory balance (ΔP − V) | Padrón + MNP 1998–2024 |
| p1c | Comparison of p1a and p1b results | — |

### Temporal coverage

MNP municipal data are available from **1998** onwards. The estimated migratory
balance series therefore covers **1998–2024**, one year shorter than p1a (1997–2024).

### Data sources

Vegetative balance data are retrieved directly from the INE API (*Resumen municipal
de fenómenos demográficos*) using the table identifiers (tpx) for each year.
No manual downloads required.

### Inputs

| File | Location | Description |
|------|----------|-------------|
| `01_padron_clean_1996_2025.csv` | `data/demography/processed/` | Municipal population 1996–2025, long format |
| `p0_municipios_goerlich_admin_hierarchy.csv` | `data/spatial/processed/` | Municipal typology + administrative hierarchy |
| INE API (MNP) | Live download | Vegetative balance by municipality, 1998–2024 |

In [ ]:
"""
Notebook  : p1b_estimated_migratory_balance.ipynb
Author    : Juan Zotes
Created   : 2026-03-25

Purpose:
    Compute estimated migratory balance (net padrón change minus vegetative
    balance from MNP) for Spanish municipalities 1998–2024, stratified by
    size group and Goerlich (2016) functional typology. Allows direct
    comparison with the EVR-based 2018 inflection point reported by
    MITERD (2022).

Inputs:
    - 01_padron_clean_1996_2025.csv              (demography/processed)
    - p0_municipios_goerlich_admin_hierarchy.csv (spatial/processed)
    - INE API: MNP tpx per year (live download, no manual files needed)

Outputs:
    Figures (figures/p1b_migratorio/):
        - p1b_fig1_national_mig_balance_{lang}.png
        - p1b_fig2_size_combined_{lang}.png
        - p1b_fig2_size_pair_{top/mid/bot}_{lang}.png
        - p1b_fig2_size_individual_{group}_{lang}.png
        - p1b_fig3_typology_combined_{lang}.png
        - p1b_fig3_typology_pair_{top/mid/bot}_{lang}.png
        - p1b_fig3_typology_individual_{type}_{lang}.png

    Derived data (demography/derived/):
        - p1b_annual_mig_balance_national.csv
        - p1b_annual_mig_balance_by_size.csv
        - p1b_annual_mig_balance_by_typology.csv
        - p1b_mnp_vegetative_balance_raw.csv     (raw MNP download, all municipalities)

Notes:
    - Vegetative balance = crecimiento vegetativo (INE MNP)
    - Estimated migratory balance = ΔP_padron − vegetative balance
    - Series starts 1998 (first MNP year available at municipal scale)
    - Size classification based on 2020 population (consistent with p1a)
    - Output CSV structure identical to p1a for direct use in p1c comparison
"""

---
## 0 · Environment and paths

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

print("Libraries loaded.")

In [ ]:
# --- Paths -----------------------------------------------------------
BASE_DIR = Path(
    r"C:\Users\juanz\OneDrive\Desktop\UCM\RURIM ESCAPE"
    r"\GeoSpatial\01_Python Data Analysis\rural-migration-land-use-spain"
)

DEMO_PROC    = BASE_DIR / "data" / "demography" / "processed"
DEMO_DERIV   = BASE_DIR / "data" / "demography" / "derived" / "paper1"
SPATIAL_PROC = BASE_DIR / "data" / "spatial" / "processed"
FIGURES_DIR  = BASE_DIR / "figures" / "p1b_migratorio"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FP_PADRON   = DEMO_PROC   / "01_padron_clean_1996_2025.csv"
FP_GOERLICH = SPATIAL_PROC / "p0_municipios_goerlich_admin_hierarchy.csv"

print("Paths defined.")
for p in [FP_PADRON, FP_GOERLICH]:
    print(f"  {'OK' if p.exists() else 'MISSING'} → {p.name}")

In [ ]:
# --- Colour palettes (identical to p1a for visual consistency) -------
GOERLICH_COLORS = {
    "Rural - Accesible"    : "#74c476",
    "Rural - Remoto"       : "#238b45",
    "Intermedio - Abierto" : "#6baed6",
    "Intermedio - Cerrado" : "#2171b5",
    "Urbano - Abierto"     : "#fd8d3c",
    "Urbano - Cerrado"     : "#bd0026",
}

TYPOLOGY_ORDER = [
    "Urbano - Cerrado",
    "Urbano - Abierto",
    "Intermedio - Cerrado",
    "Intermedio - Abierto",
    "Rural - Accesible",
    "Rural - Remoto",
]

SIZE_ORDER = [
    "> 50,000",
    "10,000 – 50,000",
    "5,000 – 10,000",
    "< 5,000 (total)",
    "1,000 – 5,000",
    "< 1,000",
]

SIZE_COLORS = {
    "> 50,000"        : "#bd0026",
    "10,000 – 50,000" : "#fd8d3c",
    "5,000 – 10,000"  : "#fecc5c",
    "< 5,000 (total)" : "#2171b5",
    "1,000 – 5,000"   : "#6baed6",
    "< 1,000"         : "#238b45",
}

COL_POS = "#2171b5"
COL_NEG = "#525252"

print("Palettes defined.")

---
## 1 · Download vegetative balance from INE MNP API

Each year has a unique table identifier (tpx) in the INE API. We iterate over all
years, filter for `crecimiento vegetativo`, and extract municipality code and value.

The municipality code is embedded in the `Nombre` field as the first 5 characters
(INE code, zero-padded).

In [ ]:
# --- INE MNP tpx by year ---------------------------------------------
# Verified manually from INEbase (Resumen municipal de fenómenos demográficos)
TPX_YEARS = {
    1998: 52184, 1999: 52185, 2000: 52186, 2001: 52187,
    2002: 52188, 2003: 52189, 2004: 52191, 2005: 52192,
    2006: 52193, 2007: 52194, 2008: 52195, 2009: 52196,
    2010: 61286, 2011: 61287, 2012: 61288, 2013: 61289,
    2014: 61290, 2015: 61291, 2016: 61292, 2017: 61293,
    2018: 61294, 2019: 61295, 2020: 61296, 2021: 61297,
    2022: 61298, 2023: 71278, 2024: 76690,
}

BASE_API = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/{}?nult=1"

print(f"Years to download: {sorted(TPX_YEARS.keys())}")
print(f"Total: {len(TPX_YEARS)} years")

In [ ]:
# --- Download loop ---------------------------------------------------
# Each API call returns all municipalities for one year.
# We filter for 'crecimiento vegetativo' and parse the Mun_Code
# from the first 5 characters of the Nombre field.

records = []
errors  = []

for year, tpx in sorted(TPX_YEARS.items()):
    url = BASE_API.format(tpx)
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        year_count = 0
        for record in data:
            nombre = record.get("Nombre", "")
            if "crecimiento vegetativo" not in nombre.lower():
                continue

            # Mun_Code is the first 5 characters of Nombre
            mun_code = nombre[:5].strip().zfill(5)

            # Value is the first (and only) element of Data
            data_list = record.get("Data", [])
            if not data_list:
                continue

            valor = data_list[0].get("Valor", None)
            secreto = data_list[0].get("Secreto", False)

            # Secreto=True means the value is suppressed for confidentiality
            if secreto:
                valor = np.nan

            records.append({
                "Mun_Code"   : mun_code,
                "Year"       : year,
                "Veg_balance": valor,
            })
            year_count += 1

        print(f"  {year} → {year_count} municipalities")
        time.sleep(0.3)   # be polite to the INE server

    except Exception as e:
        print(f"  {year} → ERROR: {e}")
        errors.append(year)

mnp = pd.DataFrame(records)
print(f"\nTotal records: {len(mnp):,}")
print(f"Years with errors: {errors if errors else 'none'}")
mnp.head()

In [ ]:
# --- Consolidate merged municipalities in MNP ------------------------
# Two municipal mergers create duplicate/inconsistent codes:
#   Oza-Cesuras:      15059 + 15063 → 15902 (merged 2013)
#   Cerdedo-Cotobade: 36012 + 36049 → 36059 (codes overlap throughout)
#
# Strategy:
#   - For Oza-Cesuras: sum 15059+15063 for 1998-2012, keep 15902 for 2013-2024
#     Drop 15059 for 2013-2024 (duplicate of 15902)
#   - For Cerdedo-Cotobade: 36059 exists for all years — use it as the
#     canonical code. Drop 36012 and 36049 entirely (already summed in 36059)

# Oza-Cesuras: drop 15059 for years >= 2013 (covered by 15902)
mask_oza = (mnp["Mun_Code"] == "15059") & (mnp["Year"] >= 2013)
mnp = mnp[~mask_oza].copy()

# Oza-Cesuras: for 1998-2012, sum 15059+15063 into 15902
oza_pre = mnp[mnp["Mun_Code"].isin(["15059", "15063"]) &
              (mnp["Year"] <= 2012)].copy()
oza_summed = (oza_pre.groupby("Year")["Veg_balance"]
              .sum(min_count=1).reset_index())
oza_summed["Mun_Code"] = "15902"
mnp = mnp[~mnp["Mun_Code"].isin(["15059", "15063"])].copy()
mnp = pd.concat([mnp, oza_summed], ignore_index=True)

# Cerdedo-Cotobade: 36059 is already the canonical code for all years
# Drop 36012 and 36049 entirely
mnp = mnp[~mnp["Mun_Code"].isin(["36012", "36049"])].copy()

print(f"MNP after consolidation: {mnp['Mun_Code'].nunique():,} municipalities")

# Verify
remaining = mnp[mnp["Mun_Code"].isin(
    ["15059","15063","15902","36012","36049","36059"]
)].sort_values(["Mun_Code","Year"])
print(remaining[["Mun_Code","Year","Veg_balance"]].to_string(index=False))

In [ ]:
# --- Save raw MNP download -------------------------------------------
fp_mnp_raw = DEMO_DERIV / "p1b_mnp_vegetative_balance_raw.csv"
mnp.to_csv(fp_mnp_raw, index=False, sep=";", encoding="utf-8-sig")
print(f"Saved → {fp_mnp_raw.name}")
print(f"Years covered: {sorted(mnp['Year'].unique())}")
print(f"Municipalities: {mnp['Mun_Code'].nunique():,}")
print(f"NaN values (suppressed): {mnp['Veg_balance'].isna().sum():,}")

---
## 2 · Load padrón and Goerlich typology

In [ ]:
# --- Load padrón (Total rows only) -----------------------------------
padron = pd.read_csv(FP_PADRON, dtype={"Mun_Code": str}, encoding="UTF-8")
padron["Mun_Code"] = padron["Mun_Code"].str.zfill(5)
padron_total = padron[padron["Cat"] == "Total"].copy()

print(f"Padrón loaded  : {len(padron_total):,} rows")
print(f"Municipalities : {padron_total['Mun_Code'].nunique():,}")

In [ ]:
# --- Load Goerlich typology ------------------------------------------
goerlich = pd.read_csv(
    FP_GOERLICH, dtype={"Mun_Code": str}, sep=";", encoding="utf-8-sig"
)
goerlich["Mun_Code"] = goerlich["Mun_Code"].str.zfill(5)

print(f"Goerlich loaded: {len(goerlich):,} rows")
print(goerlich["tipo_goerlich"].value_counts())

In [ ]:
# --- Size group classification (2020 reference year) -----------------
REF_YEAR = 2020

pop_ref = (
    padron_total[padron_total["Year"] == REF_YEAR][["Mun_Code", "Pop"]]
    .rename(columns={"Pop": "Pop_ref"})
)

def assign_size_group(pop):
    if   pop <  1_000:  return "< 1,000"
    elif pop <  5_000:  return "1,000 – 5,000"
    elif pop < 10_000:  return "5,000 – 10,000"
    elif pop < 50_000:  return "10,000 – 50,000"
    else:               return "> 50,000"

pop_ref["size_group"] = pop_ref["Pop_ref"].apply(assign_size_group)
print(f"Size groups (ref. {REF_YEAR}):")
print(pop_ref["size_group"].value_counts())

---
## 3 · Compute annual net padrón change and merge with MNP

We compute $\Delta P_{i,t}$ from the padrón (as in p1a), then merge with the
vegetative balance from MNP to obtain the estimated migratory balance:

$$\hat{M}_{i,t} = \Delta P_{i,t} - V_{i,t}$$

Note: this operation is only valid for years where both padrón and MNP data
are available (1998–2024). Year 1997 is dropped.

In [ ]:
# --- Compute ΔP from padrón ------------------------------------------
df = padron_total.sort_values(["Mun_Code", "Year"]).reset_index(drop=True)

df["Pop_prev"]  = df.groupby("Mun_Code")["Pop"].shift(1)
df["Year_prev"] = df.groupby("Mun_Code")["Year"].shift(1)
df["Year_gap"]  = df["Year"] - df["Year_prev"]
df["Delta_P"]   = df["Pop"] - df["Pop_prev"]

# Nullify non-annual transitions
df.loc[df["Year_gap"] != 1, "Delta_P"] = np.nan
df = df.drop(columns=["Pop_prev", "Year_prev", "Year_gap"])

# Merge metadata
df = df.merge(pop_ref[["Mun_Code", "Pop_ref", "size_group"]],
              on="Mun_Code", how="left")
df = df.merge(goerlich[["Mun_Code", "tipo_goerlich"]],
              on="Mun_Code", how="left")

print(f"Padrón with ΔP: {len(df):,} rows")
print(f"NaN ΔP (gaps): {df['Delta_P'].isna().sum():,}")

In [ ]:
# --- Merge MNP vegetative balance ------------------------------------
df = df.merge(mnp[["Mun_Code", "Year", "Veg_balance"]],
              on=["Mun_Code", "Year"], how="left")

# Compute estimated migratory balance
df["Mig_balance"] = df["Delta_P"] - df["Veg_balance"]

# Keep only years with MNP data (1998–2024)
df_mig = df[df["Year"] >= 1998].copy()

print(f"Rows with migratory balance: {len(df_mig):,}")
print(f"NaN Mig_balance: {df_mig['Mig_balance'].isna().sum():,}")
print(f"Years: {sorted(df_mig['Year'].unique())}")

---
## 4 · Aggregate series

In [ ]:
# --- Helper: clean aggregate -----------------------------------------
def agg_mig(frame, group_col=None):
    """Aggregate Mig_balance by Year (and optionally by group_col)."""
    if group_col:
        return (
            frame.groupby(["Year", group_col])["Mig_balance"]
            .sum(min_count=1).reset_index()
            .rename(columns={"Mig_balance": "Net_change"})
            .dropna(subset=["Net_change"])
        )
    return (
        frame.groupby("Year")["Mig_balance"]
        .sum(min_count=1).reset_index()
        .rename(columns={"Mig_balance": "Net_change"})
        .dropna(subset=["Net_change"])
    )


# National
national = agg_mig(df_mig)
print(f"National series: {len(national)} years")

# By size group
by_size = agg_mig(df_mig.dropna(subset=["size_group"]),
                  group_col="size_group")

# <5,000 aggregate
df_small = df_mig[df_mig["Pop_ref"] < 5000].copy()
small_agg = agg_mig(df_small)
small_agg["size_group"] = "< 5,000 (total)"

by_size_full = pd.concat(
    [by_size, small_agg[["Year", "Net_change", "size_group"]]],
    ignore_index=True
)

# By Goerlich typology
by_typology = agg_mig(df_mig.dropna(subset=["tipo_goerlich"]),
                      group_col="tipo_goerlich")

print(f"Size groups    : {by_size_full['size_group'].unique()}")
print(f"Typology groups: {by_typology['tipo_goerlich'].unique()}")

In [ ]:
# --- Save derived CSVs -----------------------------------------------
# Structure identical to p1a for direct use in p1c comparison
national.to_csv(DEMO_DERIV / "p1b_annual_mig_balance_national.csv",
                index=False, sep=";", encoding="utf-8-sig")
by_size_full.to_csv(DEMO_DERIV / "p1b_annual_mig_balance_by_size.csv",
                    index=False, sep=";", encoding="utf-8-sig")
by_typology.to_csv(DEMO_DERIV / "p1b_annual_mig_balance_by_typology.csv",
                   index=False, sep=";", encoding="utf-8-sig")

print("All derived CSVs saved to data/demography/derived/")

---
## 5 · Figure 1 — National aggregate (1998–2024)

In [ ]:
FIG1_LABELS = {
    "en": {
        "ylabel" : "Estimated migratory balance (thousands)",
        "xlabel" : "Year",
        "title"  : "Estimated annual migratory balance — All Spanish municipalities (1999–2024)",
        "gain"   : "Net in-migration",
        "loss"   : "Net out-migration",
        "note"   : "Estimated migratory balance = net padrón change − vegetative balance (MNP)",
    },
    "es": {
        "ylabel" : "Saldo migratorio estimado (miles)",
        "xlabel" : "Año",
        "title"  : "Saldo migratorio anual estimado — Municipios españoles (1999–2024)",
        "gain"   : "Saldo positivo",
        "loss"   : "Saldo negativo",
        "note"   : "Saldo migratorio estimado = variación neta padrón − crecimiento vegetativo (MNP)",
    },
}

for lang, labels in FIG1_LABELS.items():
    years  = national["Year"].tolist()
    values = national["Net_change"].values

    fig, ax = plt.subplots(figsize=(13, 5))
    bar_colors = [COL_POS if v >= 0 else COL_NEG for v in values]
    ax.bar(years, values / 1000, color=bar_colors, edgecolor="none", width=0.75)
    ax.axhline(0, color="black", linewidth=0.8)

    ax.set_ylabel(labels["ylabel"], fontsize=12)
    ax.set_xlabel(labels["xlabel"], fontsize=12)
    ax.set_title(labels["title"], fontsize=14, fontweight="bold")
    ax.set_xticks(years)
    ax.set_xticklabels(years, rotation=45, fontsize=10)
    ax.tick_params(axis="y", labelsize=11)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}k"))
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(handles=[
        mpatches.Patch(color=COL_POS, label=labels["gain"]),
        mpatches.Patch(color=COL_NEG, label=labels["loss"]),
    ], fontsize=11)
    ax.text(0.01, -0.15, labels["note"], transform=ax.transAxes,
            fontsize=8, color="#555555", style="italic")

    plt.tight_layout()
    fp = FIGURES_DIR / f"p1b_fig1_national_mig_balance_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if lang == "en":
        plt.show()
    else:
        plt.close()
    print(f"Saved → {fp.name}")

---
## 6 · Figure 2 — By municipality size group (1998–2024)

In [ ]:
SIZE_LABELS = {
    "en": {
        "ylabel"   : "Mig. balance (k)",
        "gain"     : "Net in-migration",
        "loss"     : "Net out-migration",
        "col_word" : "colour",
        "grey_word": "grey",
        "ref_note" : "Size classification based on 2020 population  |  "
                     "Estimated migratory balance = ΔP padrón − vegetative balance (MNP)",
        "suptitle" : "Estimated annual migratory balance by municipality size group — Spain (1998–2024)",
    },
    "es": {
        "ylabel"   : "Saldo migrat. (miles)",
        "gain"     : "Saldo positivo",
        "loss"     : "Saldo negativo",
        "col_word" : "color",
        "grey_word": "gris",
        "ref_note" : "Clasificación por tamaño basada en población de 2020  |  "
                     "Saldo migratorio estimado = ΔP padrón − crecimiento vegetativo (MNP)",
        "suptitle" : "Saldo migratorio anual estimado por grupo de tamaño — España (1998–2024)",
    },
}

SIZE_TITLES_ES = {
    "> 50,000"        : "> 50.000",
    "10,000 – 50,000" : "10.000 – 50.000",
    "5,000 – 10,000"  : "5.000 – 10.000",
    "< 5,000 (total)" : "< 5.000 (total)",
    "1,000 – 5,000"   : "1.000 – 5.000",
    "< 1,000"         : "< 1.000",
}

def draw_size_panel(ax, grp, lang, labels):
    sub    = by_size_full[by_size_full["size_group"] == grp].sort_values("Year")
    if sub.empty:
        ax.set_visible(False)
        return
    years  = sub["Year"].tolist()
    values = sub["Net_change"].values
    color  = SIZE_COLORS[grp]
    bar_colors = [color if v >= 0 else COL_NEG for v in values]
    ax.bar(years, values / 1000, color=bar_colors, edgecolor="none", width=0.75)
    ax.axhline(0, color="black", linewidth=0.6)
    if grp == "< 5,000 (total)":
        for spine in ax.spines.values():
            spine.set_linewidth(2)
            spine.set_edgecolor(color)
    panel_title = SIZE_TITLES_ES[grp] if lang == "es" else grp
    ax.set_title(panel_title, fontsize=12, fontweight="bold", color=color)
    ax.set_ylabel(labels["ylabel"], fontsize=11)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.tick_params(axis="both", labelsize=10)
    ax.set_xticks(years[::2])
    ax.set_xticklabels(years[::2], rotation=45, fontsize=10)


for lang, labels in SIZE_LABELS.items():
    col_word  = labels["col_word"]
    grey_word = labels["grey_word"]
    subtitle  = f"{labels['ref_note']}"
    show_combined = (lang == "en")

    # A) Combined 3x2
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
    for i, grp in enumerate(SIZE_ORDER):
        draw_size_panel(axes.flatten()[i], grp, lang, labels)
    fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    fp = FIGURES_DIR / f"p1b_fig2_size_combined_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if show_combined: plt.show()
    else: plt.close()
    print(f"Saved → {fp.name}")

    # B) Pairs
    for pair_idx, pair_name in enumerate(["top", "mid", "bot"]):
        grp_a = SIZE_ORDER[pair_idx * 2]
        grp_b = SIZE_ORDER[pair_idx * 2 + 1]
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
        draw_size_panel(axes[0], grp_a, lang, labels)
        draw_size_panel(axes[1], grp_b, lang, labels)
        fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                     fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        fp = FIGURES_DIR / f"p1b_fig2_size_pair_{pair_name}_{lang}.png"
        plt.savefig(fp, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved → {fp.name}")

    # C) Individual
    for grp in SIZE_ORDER:
        fig, ax = plt.subplots(figsize=(8, 5))
        draw_size_panel(ax, grp, lang, labels)
        fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                     fontsize=12, fontweight="bold", y=1.02)
        plt.tight_layout()
        slug = grp.replace(",", "").replace(" ", "_").replace(">", "gt").replace("<", "lt")
        fp = FIGURES_DIR / f"p1b_fig2_size_individual_{slug}_{lang}.png"
        plt.savefig(fp, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved → {fp.name}")

print("\nAll Figure 2 variants saved.")

---
## 7 · Figure 3 — By Goerlich (2016) typology (1998–2024)

In [ ]:
TYPOLOGY_LABELS = {
    "en": {
        "ylabel"   : "Mig. balance (k)",
        "gain"     : "Net in-migration",
        "loss"     : "Net out-migration",
        "col_word" : "colour",
        "grey_word": "grey",
        "suptitle" : "Estimated annual migratory balance by Goerlich (2016) typology — Spain (1998–2024)",
        "note"     : "Estimated migratory balance = ΔP padrón − vegetative balance (MNP)",
    },
    "es": {
        "ylabel"   : "Saldo migrat. (miles)",
        "gain"     : "Saldo positivo",
        "loss"     : "Saldo negativo",
        "col_word" : "color",
        "grey_word": "gris",
        "suptitle" : "Saldo migratorio anual estimado por tipología Goerlich (2016) — España (1998–2024)",
        "note"     : "Saldo migratorio estimado = ΔP padrón − crecimiento vegetativo (MNP)",
    },
}

TYPOLOGY_TITLES_EN = {
    "Urbano - Cerrado"     : "Urban - Closed",
    "Urbano - Abierto"     : "Urban - Open",
    "Intermedio - Cerrado" : "Intermediate - Closed",
    "Intermedio - Abierto" : "Intermediate - Open",
    "Rural - Accesible"    : "Rural - Accessible",
    "Rural - Remoto"       : "Rural - Remote",
}

TYPOLOGY_TITLES_ES = {
    "Urbano - Cerrado"     : "Urbano - Cerrado",
    "Urbano - Abierto"     : "Urbano - Abierto",
    "Intermedio - Cerrado" : "Intermedio - Cerrado",
    "Intermedio - Abierto" : "Intermedio - Abierto",
    "Rural - Accesible"    : "Rural - Accesible",
    "Rural - Remoto"       : "Rural - Remoto",
}

def draw_typology_panel(ax, typ, lang, labels):
    sub    = by_typology[by_typology["tipo_goerlich"] == typ].sort_values("Year")
    if sub.empty:
        ax.set_visible(False)
        return
    years  = sub["Year"].tolist()
    values = sub["Net_change"].values
    color  = GOERLICH_COLORS[typ]
    bar_colors = [color if v >= 0 else COL_NEG for v in values]
    ax.bar(years, values / 1000, color=bar_colors, edgecolor="none", width=0.75)
    ax.axhline(0, color="black", linewidth=0.6)
    panel_title = TYPOLOGY_TITLES_EN[typ] if lang == "en" else TYPOLOGY_TITLES_ES[typ]
    ax.set_title(panel_title, fontsize=12, fontweight="bold", color=color)
    ax.set_ylabel(labels["ylabel"], fontsize=11)
    ax.grid(axis="y", linestyle=":", alpha=0.4)
    ax.tick_params(axis="both", labelsize=10)
    ax.set_xticks(years[::2])
    ax.set_xticklabels(years[::2], rotation=45, fontsize=10)


for lang, labels in TYPOLOGY_LABELS.items():
    subtitle      = labels["note"]
    show_combined = (lang == "en")

    # A) Combined 3x2
    fig, axes = plt.subplots(3, 2, figsize=(14, 12), sharex=True)
    for i, typ in enumerate(TYPOLOGY_ORDER):
        draw_typology_panel(axes.flatten()[i], typ, lang, labels)
    fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    fp = FIGURES_DIR / f"p1b_fig3_typology_combined_{lang}.png"
    plt.savefig(fp, dpi=300, bbox_inches="tight")
    if show_combined: plt.show()
    else: plt.close()
    print(f"Saved → {fp.name}")

    # B) Pairs
    for pair_idx, pair_name in enumerate(["top", "mid", "bot"]):
        typ_a = TYPOLOGY_ORDER[pair_idx * 2]
        typ_b = TYPOLOGY_ORDER[pair_idx * 2 + 1]
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
        draw_typology_panel(axes[0], typ_a, lang, labels)
        draw_typology_panel(axes[1], typ_b, lang, labels)
        fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                     fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        fp = FIGURES_DIR / f"p1b_fig3_typology_pair_{pair_name}_{lang}.png"
        plt.savefig(fp, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved → {fp.name}")

    # C) Individual
    for typ in TYPOLOGY_ORDER:
        fig, ax = plt.subplots(figsize=(8, 5))
        draw_typology_panel(ax, typ, lang, labels)
        fig.suptitle(f"{labels['suptitle']}\n{subtitle}",
                     fontsize=12, fontweight="bold", y=1.02)
        plt.tight_layout()
        slug = typ.replace(" ", "_").replace("-", "").replace("__", "_")
        fp = FIGURES_DIR / f"p1b_fig3_typology_individual_{slug}_{lang}.png"
        plt.savefig(fp, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved → {fp.name}")

print("\nAll Figure 3 variants saved.")

#### 8 · Interpretation notes

**Key findings from this notebook (p1b):**

- **Municipalities <5,000 hab.:** the estimated migratory balance becomes
  sustainably positive from **2018**, confirming the MITERD (2022) inflection
  point independently. This removes the ~2-year lag observed in p1a and
  demonstrates that the lag was entirely attributable to the structural
  negative vegetative balance of small rural municipalities.
- **Vegetative balance lag:** removing the vegetative component shifts the
  inflection point forward by approximately 2 years (from ~2020 in p1a
  to 2018 in p1b), consistent with the hypothesis stated in p1a.
- **Rural Accesible vs Rural Remoto:** both show the same 2018 inflection
  in the migratory balance. However, Rural Remoto shows greater volatility
  and lower absolute magnitudes, reflecting its structural fragility. The
  recovery signal is real in both but weaker and less sustained in Rural Remoto.
- **Urban typologies (COVID effect):** Urban Closed and Urban Open show a
  sharp drop in migratory balance in 2020–2021, consistent with the
  well-documented urban exodus during the COVID pandemic. This is more
  pronounced in the migratory balance (p1b) than in the net padrón change
  (p1a), where excess COVID mortality partially offset the migratory signal.
  This is a methodologically important finding — the two measures diverge
  precisely where COVID excess mortality was concentrated (urban areas).
- **National series:** the estimated migratory balance remained positive
  throughout most of the series, turning negative only in 2013–2015 at the
  peak of the economic crisis. The 2016–2017 recovery precedes the rural
  inflection, suggesting the national migratory recovery was led initially
  by urban and intermediate municipalities before reaching rural ones.

**Key questions addressed:**

1. ✓ Confirmed — migratory balance for <5,000 hab. becomes positive in 2018.
2. ✓ The vegetative lag is ~2 years, consistent across size groups and typologies.
3. ✓ Rural Remoto and Rural Accesible share the 2018 inflection but differ in magnitude and volatility.
4. ✓ Urban typologies show sustained positive migratory balance except 2020–2022 (COVID urban exodus).
5. ✓ The COVID effect is more visible in p1b than p1a for urban areas — divergence between measures is interpretable.

## 9 · Dependencies

All packages are standard and available in the `rural-migration` conda environment:
```
pandas, numpy, matplotlib, requests
```

No additional installs required for this notebook.